# 02. Prompting Agéntico

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 75 minutos  
**Prerequisitos:** [01. Introducción a LLM Agents](01-intro-llm-agents.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Implementar Chain-of-Thought (CoT) prompting para razonamiento paso a paso
- Usar Tree-of-Thought (ToT) para explorar múltiples caminos de razonamiento
- Aplicar Self-Consistency para mejorar confiabilidad de respuestas
- Diseñar prompts few-shot efectivos para guiar comportamiento del agente
- Comparar y contrastar diferentes estrategias de prompting

## 1. Motivación: El Poder del Prompting

### El Problema: LLMs Pueden "Pensar" Mal

Pregunta simple: *"Roger tiene 5 pelotas de tenis. Compra 2 latas más de pelotas. Cada lata tiene 3 pelotas. ¿Cuántas pelotas tiene ahora?"*

**LLM sin prompting especial:**
```
Respuesta: Roger tiene 10 pelotas.
```
❌ **Incorrecto** (5 + 2*3 = 11)

**LLM con Chain-of-Thought:**
```
Pensemos paso a paso:
1. Roger empieza con 5 pelotas
2. Compra 2 latas, cada lata tiene 3 pelotas
3. Pelotas nuevas: 2 × 3 = 6 pelotas
4. Total: 5 + 6 = 11 pelotas

Respuesta: Roger tiene 11 pelotas.
```
✅ **Correcto**

### La Solución: Prompting Agéntico

Técnicas que hacen que LLMs "piensen" mejor:

1. **Chain-of-Thought (CoT)**: Razonamiento secuencial explícito
2. **Tree-of-Thought (ToT)**: Exploración ramificada de posibilidades
3. **Self-Consistency**: Múltiples intentos + votación
4. **Few-Shot Learning**: Aprender de ejemplos concretos

### Pregunta Guía

**Al final de este notebook responderemos:**
*¿Cómo podemos hacer que un LLM razone de forma más confiable y sistemática usando solo el diseño del prompt?*

## 2. Intuición Visual: Estrategias de Razonamiento

### Comparación Visual

```
┌──────────────────────────────────────────────────────────┐
│              PROMPTING ESTÁNDAR (Naive)                 │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Pregunta ──────► [LLM] ──────► Respuesta              │
│                                                          │
│  • Una sola pasada                                      │
│  • Sin razonamiento explícito                           │
│  • Propenso a errores                                   │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              CHAIN-OF-THOUGHT (CoT)                     │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Pregunta ──► [LLM] ──► Paso 1 ──► Paso 2 ──► ... ──►  │
│                           ↓          ↓                   │
│                        Explícito  Explícito              │
│                                      ↓                   │
│                                  Respuesta               │
│                                                          │
│  • Razonamiento paso a paso                             │
│  • Trazabilidad completa                                │
│  • Más confiable                                        │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              TREE-OF-THOUGHT (ToT)                      │
├──────────────────────────────────────────────────────────┤
│                                                          │
│                      Pregunta                            │
│                         │                                │
│              ┌──────────┼──────────┐                    │
│              ▼          ▼          ▼                    │
│           Camino A   Camino B   Camino C                │
│              │          │          │                    │
│         ┌────┴────┐    │     ┌────┴────┐               │
│         ▼         ▼    ▼     ▼         ▼               │
│       Sub-A1   Sub-A2  ...  Sub-C1   Sub-C2            │
│                                                          │
│  [Evaluación y selección del mejor camino]             │
│                         │                                │
│                         ▼                                │
│                    Respuesta                             │
│                                                          │
│  • Exploración de alternativas                          │
│  • Evaluación de caminos                                │
│  • Backtracking si es necesario                         │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              SELF-CONSISTENCY                           │
├──────────────────────────────────────────────────────────┤
│                                                          │
│                     Pregunta                             │
│                        │                                 │
│       ┌────────────────┼────────────────┐               │
│       │                │                │               │
│       ▼                ▼                ▼               │
│   [LLM CoT]       [LLM CoT]       [LLM CoT]            │
│   (temp=0.7)      (temp=0.7)      (temp=0.7)           │
│       │                │                │               │
│       ▼                ▼                ▼               │
│  Respuesta A    Respuesta B    Respuesta C              │
│                        │                                 │
│                   [VOTACIÓN]                             │
│                        │                                 │
│                        ▼                                 │
│                Respuesta Final                           │
│                (mayoría gana)                            │
│                                                          │
│  • Múltiples ejecuciones                                │
│  • Mayor confiabilidad                                  │
│  • Costoso (N × llamadas)                               │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# Instalación de dependencias
# !pip install openai anthropic python-dotenv plotly numpy

import os
import json
import re
from typing import List, Dict, Tuple, Optional
from collections import Counter
from dataclasses import dataclass
import numpy as np

# Visualizaciones
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# APIs
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible")

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 3. Fundamentos Matemáticos: Teoría del Prompting

### Modelando el Prompting

Sea $\mathcal{L}$ un LLM, formalmente podemos expresar:

$$
\begin{align}
y &= \mathcal{L}(x) \tag{1} \\
\text{donde: } & \\
x &: \text{prompt (input)} \\
y &: \text{respuesta (output)} \\
\mathcal{L} &: \text{función LLM} 
\end{align}
$$

### Chain-of-Thought (CoT)

En lugar de $x \to y$ directamente, descomponemos:

$$
\begin{align}
x &\to r_1 \to r_2 \to ... \to r_n \to y \tag{2} \\
\text{donde: } & \\
r_i &: \text{paso intermedio de razonamiento} \\
n &: \text{número de pasos}
\end{align}
$$

El prompt CoT típicamente incluye: $x_{\text{CoT}} = x + \text{"Let's think step by step"}$

### Self-Consistency

Generamos $k$ respuestas con temperatura $T > 0$:

$$
\begin{align}
\{y_1, y_2, ..., y_k\} &= \{\mathcal{L}(x, T)_i\}_{i=1}^k \tag{3} \\
y_{\text{final}} &= \text{mode}(\{y_1, y_2, ..., y_k\}) \tag{4} \\
\text{donde: } & \\
\text{mode}() &: \text{valor más frecuente (votación)} 
\end{align}
$$

**Probabilidad de correctitud:**

Si cada intento tiene probabilidad $p$ de ser correcto:

$$
P(\text{correcto}) = \sum_{i=\lceil k/2 \rceil}^{k} \binom{k}{i} p^i (1-p)^{k-i} \tag{5}
$$

**Ejemplo numérico:** Si $p=0.7$ (70% accuracy individual) y $k=5$:

$$
P(\text{mayoría correcta}) \approx 0.837 \text{ (83.7%)}
$$

### Tree-of-Thought (ToT)

Modelamos como búsqueda en árbol:

$$
\begin{align}
S(s, a) &= \text{evaluación del estado } s \text{ tras acción } a \tag{6} \\
\text{best\_path} &= \arg\max_{\text{path}} \sum_{s \in \text{path}} S(s) \tag{7}
\end{align}
$$

Donde $S(s)$ puede ser evaluado por el mismo LLM o heurísticas.

**Key Insight:**
> El prompting no cambia las capacidades del modelo, pero sí cómo las usa. Es como la diferencia entre "pensar en voz alta" vs "responder impulsivamente".

## 4. Implementación Desde Cero

### 4.1 Chain-of-Thought (CoT)

In [ ]:
class ChainOfThoughtPrompting:
    """
    Implementación de Chain-of-Thought prompting.
    """
    
    def __init__(self, llm_backend="simulated", model="gpt-4-turbo-preview"):
        self.llm_backend = llm_backend
        self.model = model
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        else:
            self.client = None
    
    def _call_llm(self, prompt: str, temperature: float = 0.0) -> str:
        """Llama al LLM"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature
            )
            return response.choices[0].message.content
        else:
            # Simulado
            return self._simulated_cot_response(prompt)
    
    def _simulated_cot_response(self, prompt: str) -> str:
        """Simula una respuesta CoT (solo para demo)"""
        if "pelotas" in prompt.lower():
            return """Pensemos paso a paso:
1. Roger empieza con 5 pelotas de tenis
2. Compra 2 latas más de pelotas
3. Cada lata contiene 3 pelotas
4. Pelotas en las latas: 2 × 3 = 6 pelotas
5. Total: 5 (inicial) + 6 (nuevas) = 11 pelotas

Por lo tanto, Roger tiene 11 pelotas de tenis ahora."""
        return "Respuesta simulada"
    
    def zero_shot_cot(self, question: str) -> str:
        """
        Zero-shot CoT: Solo agrega "Let's think step by step"
        
        Paper: "Large Language Models are Zero-Shot Reasoners" (Kojima et al., 2022)
        """
        prompt = f"{question}\n\nLet's think step by step:"
        return self._call_llm(prompt)
    
    def few_shot_cot(self, question: str, examples: List[Dict[str, str]]) -> str:
        """
        Few-shot CoT: Proporciona ejemplos con razonamiento
        
        Args:
            question: Pregunta a responder
            examples: Lista de {"question": ..., "reasoning": ..., "answer": ...}
        """
        # Construir prompt con ejemplos
        prompt_parts = []
        
        for i, ex in enumerate(examples, 1):
            prompt_parts.append(f"Ejemplo {i}:")
            prompt_parts.append(f"Pregunta: {ex['question']}")
            prompt_parts.append(f"Razonamiento: {ex['reasoning']}")
            prompt_parts.append(f"Respuesta: {ex['answer']}")
            prompt_parts.append("")
        
        # Agregar pregunta actual
        prompt_parts.append(f"Ahora responde esta pregunta:")
        prompt_parts.append(f"Pregunta: {question}")
        prompt_parts.append(f"Razonamiento:")
        
        prompt = "\n".join(prompt_parts)
        return self._call_llm(prompt)

# Crear instancia
cot = ChainOfThoughtPrompting(llm_backend="simulated")

# Probar Zero-Shot CoT
question = "Roger tiene 5 pelotas de tenis. Compra 2 latas más de pelotas. Cada lata tiene 3 pelotas. ¿Cuántas pelotas tiene ahora?"
response = cot.zero_shot_cot(question)

print("🧮 Zero-Shot Chain-of-Thought")
print("="*60)
print(f"Pregunta: {question}")
print(f"\nRespuesta:\n{response}")

### 4.2 Self-Consistency

In [ ]:
class SelfConsistency:
    """
    Implementa Self-Consistency: múltiples muestreos + votación
    
    Paper: "Self-Consistency Improves Chain of Thought Reasoning in Language Models"
    (Wang et al., 2022)
    """
    
    def __init__(self, cot_prompter: ChainOfThoughtPrompting):
        self.cot = cot_prompter
    
    def _extract_final_answer(self, response: str) -> str:
        """
        Extrae la respuesta final del razonamiento CoT.
        
        Busca patrones como:
        - "Por lo tanto, X"
        - "La respuesta es X"
        - Último número mencionado
        """
        # Buscar patrones comunes
        patterns = [
            r'por lo tanto[,:]?\s*(?:tiene|hay|es|son)?\s*(\d+)',
            r'la respuesta es\s*(\d+)',
            r'total[:\s]*(\d+)',
            r'= (\d+)\s*(?:pelotas|unidades)?\s*$'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, response.lower())
            if match:
                return match.group(1)
        
        # Fallback: último número en la respuesta
        numbers = re.findall(r'\d+', response)
        if numbers:
            return numbers[-1]
        
        return response.strip()
    
    def run(
        self, 
        question: str, 
        num_samples: int = 5,
        temperature: float = 0.7,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta Self-Consistency.
        
        Args:
            question: Pregunta a responder
            num_samples: Número de muestreos (típicamente 5-40)
            temperature: Temperatura para sampling (típicamente 0.7)
            verbose: Imprimir pasos
        
        Returns:
            Dict con respuesta final, todas las respuestas, y conteos
        """
        if verbose:
            print(f"\n🔄 Self-Consistency con {num_samples} muestreos")
            print("="*60)
        
        # Generar múltiples razonamientos
        all_responses = []
        all_answers = []
        
        for i in range(num_samples):
            if verbose:
                print(f"\n--- Muestreo {i+1}/{num_samples} ---")
            
            # Generar respuesta con CoT
            response = self.cot.zero_shot_cot(question)
            all_responses.append(response)
            
            # Extraer respuesta final
            answer = self._extract_final_answer(response)
            all_answers.append(answer)
            
            if verbose:
                print(f"Respuesta extraída: {answer}")
        
        # Votar por mayoría
        answer_counts = Counter(all_answers)
        most_common_answer, count = answer_counts.most_common(1)[0]
        
        if verbose:
            print(f"\n📊 Votación:")
            for answer, cnt in answer_counts.most_common():
                print(f"  {answer}: {cnt}/{num_samples} votos ({cnt/num_samples*100:.1f}%)")
            print(f"\n✅ Respuesta final (mayoría): {most_common_answer}")
        
        return {
            "final_answer": most_common_answer,
            "all_responses": all_responses,
            "all_answers": all_answers,
            "vote_counts": dict(answer_counts),
            "confidence": count / num_samples
        }

# Probar Self-Consistency
sc = SelfConsistency(cot)
result = sc.run(question, num_samples=5, verbose=True)

print(f"\n🎯 Confianza: {result['confidence']*100:.1f}%")

### 4.3 Tree-of-Thought (ToT)

In [ ]:
class TreeOfThought:
    """
    Implementación simplificada de Tree-of-Thought.
    
    Paper: "Tree of Thoughts: Deliberate Problem Solving with LLMs" (Yao et al., 2023)
    
    ToT permite explorar múltiples caminos de razonamiento y hacer backtracking.
    """
    
    def __init__(self, cot_prompter: ChainOfThoughtPrompting):
        self.cot = cot_prompter
    
    def generate_thoughts(self, problem: str, num_thoughts: int = 3) -> List[str]:
        """
        Genera múltiples "pensamientos" iniciales (caminos posibles).
        """
        prompt = f"""Dado el siguiente problema, genera {num_thoughts} enfoques diferentes para resolverlo.

Problema: {problem}

Lista {num_thoughts} enfoques distintos (solo el primer paso de cada uno):"""
        
        response = self.cot._call_llm(prompt)
        
        # Parsear respuestas (simplificado)
        thoughts = []
        for line in response.split('\n'):
            if line.strip() and (line[0].isdigit() or line.startswith('-')):
                thought = re.sub(r'^[\d.-]+\s*', '', line.strip())
                if thought:
                    thoughts.append(thought)
        
        # Si no se parseó bien, retornar respuesta completa
        if not thoughts:
            thoughts = [f"Enfoque {i+1}": {problem}" for i in range(num_thoughts)]
        
        return thoughts[:num_thoughts]
    
    def evaluate_thought(self, problem: str, thought: str) -> float:
        """
        Evalúa qué tan prometedor es un pensamiento (0-1).
        
        En la práctica, esto podría ser:
        - Evaluación por el LLM
        - Heurística de dominio
        - Modelo separado
        """
        prompt = f"""Evalúa qué tan prometedor es el siguiente enfoque para resolver el problema.

Problema: {problem}

Enfoque: {thought}

Da una calificación de 1-10 donde:
- 1: Muy poco prometedor
- 10: Muy prometedor

Calificación (solo el número):"""
        
        response = self.cot._call_llm(prompt)
        
        # Extraer número
        match = re.search(r'(\d+)', response)
        if match:
            score = int(match.group(1))
            return min(score / 10.0, 1.0)
        
        return 0.5  # Default
    
    def run(
        self,
        problem: str,
        num_thoughts: int = 3,
        depth: int = 2,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta ToT con búsqueda en árbol.
        
        Args:
            problem: Problema a resolver
            num_thoughts: Número de pensamientos a generar por nivel
            depth: Profundidad del árbol
            verbose: Imprimir pasos
        """
        if verbose:
            print(f"\n🌳 Tree-of-Thought (profundidad={depth}, breadth={num_thoughts})")
            print("="*60)
        
        # Generar pensamientos iniciales
        thoughts = self.generate_thoughts(problem, num_thoughts)
        
        if verbose:
            print(f"\n📝 Pensamientos generados:")
            for i, t in enumerate(thoughts, 1):
                print(f"  {i}. {t}")
        
        # Evaluar pensamientos
        evaluations = []
        for thought in thoughts:
            score = self.evaluate_thought(problem, thought)
            evaluations.append(score)
        
        if verbose:
            print(f"\n📊 Evaluaciones:")
            for i, (thought, score) in enumerate(zip(thoughts, evaluations), 1):
                print(f"  {i}. Score: {score:.2f} - {thought[:50]}...")
        
        # Seleccionar mejor camino
        best_idx = np.argmax(evaluations)
        best_thought = thoughts[best_idx]
        
        if verbose:
            print(f"\n✅ Mejor camino seleccionado: {best_thought}")
        
        # Continuar con el mejor camino usando CoT
        final_prompt = f"{problem}\n\nEnfoque seleccionado: {best_thought}\n\nResuelve paso a paso:"
        final_response = self.cot._call_llm(final_prompt)
        
        return {
            "all_thoughts": thoughts,
            "evaluations": evaluations,
            "best_thought": best_thought,
            "final_response": final_response
        }

# Probar ToT
tot = TreeOfThought(cot)
result = tot.run(question, num_thoughts=3, verbose=True)

print(f"\n🎯 Respuesta final:")
print(result['final_response'])

## 5. Versión con Framework: LangChain

LangChain incluye soporte para varias de estas técnicas.

In [ ]:
# Ejemplo conceptual con LangChain
# !pip install langchain langchain-openai

try:
    from langchain.prompts import PromptTemplate
    from langchain_openai import ChatOpenAI
    
    # Template para CoT
    cot_template = PromptTemplate(
        input_variables=["question"],
        template="""{question}\n\nLet's solve this step by step:\n"""
    )
    
    # LLM
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    
    # Chain
    # cot_chain = cot_template | llm
    # response = cot_chain.invoke({"question": question})
    
    print("✅ LangChain disponible")
except ImportError:
    print("⚠️  LangChain no disponible")

## 6. Visualización de Resultados

In [ ]:
def compare_prompting_strategies():
    """
    Compara diferentes estrategias de prompting.
    """
    strategies = ['Naive', 'Zero-Shot CoT', 'Few-Shot CoT', 'Self-Consistency', 'Tree-of-Thought']
    
    # Métricas simuladas (en práctica, medir en benchmark)
    accuracy = [0.65, 0.78, 0.82, 0.87, 0.85]
    cost_factor = [1.0, 1.0, 1.2, 5.0, 8.0]  # Relativo a naive
    latency_factor = [1.0, 1.1, 1.3, 5.2, 4.8]
    
    # Crear subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Accuracy', 'Cost (relative)', 'Latency (relative)', 'Accuracy vs Cost'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}],
               [{'type': 'bar'}, {'type': 'scatter'}]]
    )
    
    # Accuracy
    fig.add_trace(
        go.Bar(x=strategies, y=accuracy, marker_color='#3498db', name='Accuracy'),
        row=1, col=1
    )
    
    # Cost
    fig.add_trace(
        go.Bar(x=strategies, y=cost_factor, marker_color='#e74c3c', name='Cost'),
        row=1, col=2
    )
    
    # Latency
    fig.add_trace(
        go.Bar(x=strategies, y=latency_factor, marker_color='#f39c12', name='Latency'),
        row=2, col=1
    )
    
    # Accuracy vs Cost (Pareto frontier)
    fig.add_trace(
        go.Scatter(
            x=cost_factor, 
            y=accuracy,
            mode='markers+text',
            marker=dict(size=12, color='#2ecc71'),
            text=strategies,
            textposition="top center",
            name='Strategies'
        ),
        row=2, col=2
    )
    
    fig.update_xaxes(title_text="Strategy", row=1, col=1)
    fig.update_xaxes(title_text="Strategy", row=1, col=2)
    fig.update_xaxes(title_text="Strategy", row=2, col=1)
    fig.update_xaxes(title_text="Cost Factor", row=2, col=2)
    
    fig.update_yaxes(title_text="Accuracy", row=1, col=1, range=[0, 1])
    fig.update_yaxes(title_text="Cost Factor", row=1, col=2)
    fig.update_yaxes(title_text="Latency Factor", row=2, col=1)
    fig.update_yaxes(title_text="Accuracy", row=2, col=2, range=[0, 1])
    
    fig.update_layout(
        height=800,
        title_text="Comparación de Estrategias de Prompting",
        showlegend=False,
        template='plotly_white'
    )
    
    return fig

fig = compare_prompting_strategies()
fig.show()

## 7. Ejercicios

### 🟢 Ejercicio 1: Few-Shot Examples

Crea ejemplos few-shot efectivos para un problema de razonamiento lógico.

In [ ]:
def ejercicio_1_few_shot():
    """
    Objetivo: Diseñar ejemplos few-shot para mejorar razonamiento
    
    Tarea: Crear 2-3 ejemplos para problemas de "días de la semana"
    Ejemplo: "Si hoy es lunes y viajas 10 días, ¿qué día será?"
    """
    # TODO: Crea una lista de ejemplos
    examples = [
        {
            "question": "...",
            "reasoning": "...",
            "answer": "..."
        },
        # Agrega más ejemplos
    ]
    
    # TODO: Prueba con few_shot_cot
    # cot = ChainOfThoughtPrompting()
    # result = cot.few_shot_cot("Si hoy es miércoles y viajas 15 días, ¿qué día será?", examples)
    
    pass

# ejercicio_1_few_shot()

### 🟡 Ejercicio 2: Optimizar Self-Consistency

Experimenta con diferentes valores de `num_samples` y analiza el tradeoff accuracy vs cost.

In [ ]:
def ejercicio_2_optimize_sc():
    """
    Objetivo: Encontrar el número óptimo de samples para Self-Consistency
    
    Instrucciones:
    1. Prueba con num_samples = [1, 3, 5, 10, 20]
    2. Mide accuracy (si tienes ground truth) y confianza
    3. Grafica accuracy vs cost (cost = num_samples × cost_per_call)
    4. Identifica el "sweet spot"
    """
    # TODO: Tu código aquí
    pass

# ejercicio_2_optimize_sc()

### 🔴 Ejercicio 3: Implementar ReAct con CoT

Combina CoT con el patrón ReAct (Reasoning + Acting) del notebook anterior.

In [ ]:
def ejercicio_3_react_with_cot():
    """
    Objetivo: Integrar CoT en un agente ReAct
    
    Idea:
    - En cada paso de razonamiento, usa CoT
    - Esto debería hacer que el agente "piense mejor" sobre qué herramienta usar
    
    Desafío:
    Modifica SimpleAgent del notebook 01 para que use CoT en cada decisión.
    """
    # TODO: Tu código aquí
    # Pista: En _build_prompt(), agrega instrucciones CoT
    # Pista 2: Esto será explorado más a fondo en notebook 03
    pass

# Este ejercicio es avanzado - prepara para el siguiente notebook

## 8. Resumen y Recursos

### 📚 Resumen

- **Chain-of-Thought (CoT)**: Razonamiento paso a paso explícito mejora accuracy dramáticamente
  - Zero-shot: Solo agregar "Let's think step by step"
  - Few-shot: Proporcionar ejemplos con razonamiento
  
- **Self-Consistency**: Múltiples muestreos + votación mayoría
  - Mejora confiabilidad a costa de más llamadas al LLM
  - Típicamente 5-40 samples, sweet spot ~10-20
  
- **Tree-of-Thought (ToT)**: Búsqueda en árbol de razonamientos
  - Exploración de múltiples caminos
  - Evaluación y selección del mejor
  - Permite backtracking
  
- **Tradeoffs**:
  - Naive: Rápido pero menos preciso
  - CoT: Balance ideal para mayoría de casos
  - Self-Consistency: Máxima confiabilidad, costoso
  - ToT: Para problemas que requieren exploración

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

1. **"Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"** (Wei et al., 2022)
   - [https://arxiv.org/abs/2201.11903](https://arxiv.org/abs/2201.11903)
   - Paper seminal que introdujo CoT
   
2. **"Large Language Models are Zero-Shot Reasoners"** (Kojima et al., 2022)
   - [https://arxiv.org/abs/2205.11916](https://arxiv.org/abs/2205.11916)
   - Descubrió que "Let's think step by step" es sorprendentemente efectivo
   
3. **"Self-Consistency Improves Chain of Thought Reasoning"** (Wang et al., 2022)
   - [https://arxiv.org/abs/2203.11171](https://arxiv.org/abs/2203.11171)
   - Introduce self-consistency
   
4. **"Tree of Thoughts: Deliberate Problem Solving with LLMs"** (Yao et al., 2023)
   - [https://arxiv.org/abs/2305.10601](https://arxiv.org/abs/2305.10601)
   - Generaliza CoT a búsqueda en árbol

#### 🎥 Videos

- **"Prompt Engineering Guide"** - Elvis Saravia
- **"Advanced Prompting Techniques"** - Andrew Ng

#### 💻 Recursos

- **Prompt Engineering Guide**: [https://www.promptingguide.ai/](https://www.promptingguide.ai/)
- **OpenAI Cookbook - Prompting**: [https://cookbook.openai.com/](https://cookbook.openai.com/)

### ➡️ Próximo Paso

En el siguiente notebook **"03. ReAct (Reasoning + Acting)"**, combinaremos estas técnicas de prompting con el loop de agente del notebook 01:

- Patrón ReAct formal
- Alternancia explícita entre razonamiento y acción
- Wikipedia search como ejemplo práctico
- Trazabilidad completa de decisiones

**[➡️ Ir al Notebook 03: ReAct Pattern](03-react-reasoning-acting.ipynb)**

---

<div align="center">

### Respuesta a la Pregunta Guía

*¿Cómo hacer que un LLM razone más confiablemente?*

**Respuesta:** Diseñando prompts que **expliciten el proceso de razonamiento**:
- CoT descompone en pasos intermedios
- Self-Consistency promedia múltiples caminos
- ToT explora alternativas sistemáticamente

El prompting no mejora la inteligencia del modelo, pero sí **cómo la usa**.

</div>